# 03 - Volatility Modeling & Forecasting Pipeline

This notebook acts as an execution and inspection runner for the hybrid volatility forecasting pipeline (AR(1)-GARCH(1,1) + Sentiment-LSTM).

When running on Kaggle, this notebook will:
- Read keys (`GITHUB_TOKEN`, etc.) from Kaggle Secrets.
- Clone or pull the active branch of the GitHub repo into `/kaggle/working`.
- Materialize credentials in `.dvc/credentials/key.json` and `.dvc/config.local` for service account access.
- Pull DVC tracking targets (like raw news files, prices, and pre-computed sentiment scores).
- Install dependencies and run the volatility pipeline stages against the checked-out workspace.

## 0 - Bootstrap

In [1]:
import base64
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

STAGED_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
IS_KAGGLE = Path('/kaggle').exists()

def run_command(*args: str, cwd: str | Path | None = None, display: str | None = None) -> None:
    command = [str(arg) for arg in args]
    cmd_str = display or ' '.join(command)
    import re
    cmd_str = re.sub(r'https://[^@]+@github\.com', 'https://<TOKEN>@github.com', cmd_str)
    print('$', cmd_str)
    subprocess.run(command, check=True, cwd=cwd)

def load_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'))

def load_kaggle_secret(name: str) -> str | None:
    if not IS_KAGGLE:
        return os.getenv(name)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)

def parse_json_secret(value: str | None) -> dict | None:
    if not value:
        return None
    candidates = [value]
    try:
        decoded = base64.b64decode(value).decode('utf-8')
        candidates.append(decoded)
    except Exception:
        pass
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            continue
    return None

def write_service_account_key(secret_name: str, repo_root: Path) -> Path | None:
    secret_payload = parse_json_secret(load_kaggle_secret(secret_name))
    if not secret_payload:
        return None
    key_path = repo_root / '.dvc' / 'credentials' / 'key.json'
    key_path.parent.mkdir(parents=True, exist_ok=True)
    key_path.write_text(json.dumps(secret_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return key_path

def configure_dvc_service_account(repo_root: Path, key_path: Path, remote_name: str = 'gdrive') -> Path:
    config_local_path = repo_root / '.dvc' / 'config.local'
    rel_key_path = key_path.relative_to(repo_root / '.dvc')
    config_lines = [
        f'[remote "{remote_name}"]',
        '    gdrive_use_service_account = true',
        f'    gdrive_service_account_json_file_path = {rel_key_path.as_posix()}'
    ]
    config_local_path.write_text('\n'.join(config_lines) + '\n', encoding='utf-8')
    return config_local_path

def github_auth_url(repo_url: str, token: str | None) -> str:
    if not token or not repo_url.startswith('https://github.com/'):
        return repo_url
    return repo_url.replace('https://', f'https://{token}@', 1)

print('Running on Kaggle:', IS_KAGGLE)
print('Staged root     :', STAGED_ROOT)


Running on Kaggle: True
Staged root     : /kaggle/working


## 1 - Runtime Configuration

In [2]:
REPO_GIT_URL = 'https://github.com/tlong-ds/news-sentiment-analysis.git'
REPO_BRANCH = 'main'
REPO_DIR_NAME = 'news-sentiment-analysis'
PULL_REPO_ON_KAGGLE = IS_KAGGLE
INSTALL_PROJECT_REQUIREMENTS = IS_KAGGLE
LOAD_KAGGLE_SECRETS = IS_KAGGLE
GDRIVE_SERVICE_ACCOUNT_SECRET = 'GDRIVE_SERVICE_ACCOUNT_JSON'
SETUP_DVC_SERVICE_ACCOUNT = IS_KAGGLE
DVC_PULL_ON_KAGGLE = IS_KAGGLE
DVC_PULL_REMOTE = 'gdrive'

# We pull raw data and existing sentiment parquets to run volatility modeling
DVC_PULL_TARGETS = [
    'data/raw.dvc',
    'pipelines/sentiment/dvc.yaml'
]



## 2 - Kaggle Secrets, Repo Pull, and DVC Bootstrap

In [3]:
if LOAD_KAGGLE_SECRETS:
    for secret_name in ['GITHUB_TOKEN', 'GEMINI_API_KEY']:
        secret_value = load_kaggle_secret(secret_name)
        if secret_value and secret_name not in os.environ:
            os.environ[secret_name] = secret_value

if IS_KAGGLE and PULL_REPO_ON_KAGGLE:
    kaggle_repo_root = Path('/kaggle/working') / REPO_DIR_NAME
    auth_repo_url = github_auth_url(REPO_GIT_URL, os.getenv('GITHUB_TOKEN'))

    if kaggle_repo_root.exists() and (kaggle_repo_root / '.git').exists():
        run_command('git', 'fetch', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'checkout', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'pull', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
    else:
        run_command(
            'git', 'clone', '--branch', REPO_BRANCH, auth_repo_url, str(kaggle_repo_root),
            cwd='/kaggle/working'
        )
    ROOT = kaggle_repo_root
else:
    ROOT = STAGED_ROOT

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
importlib.invalidate_caches()

if INSTALL_PROJECT_REQUIREMENTS:
    run_command(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', cwd=ROOT)
else:
    print('Skipping pip install.')

service_account_key_path = None
dvc_config_local_path = None
if SETUP_DVC_SERVICE_ACCOUNT:
    service_account_key_path = write_service_account_key(GDRIVE_SERVICE_ACCOUNT_SECRET, ROOT)
    if service_account_key_path is None:
        print(f'{GDRIVE_SERVICE_ACCOUNT_SECRET} not found or invalid; skipping DVC service-account setup.')
    else:
        dvc_config_local_path = configure_dvc_service_account(ROOT, service_account_key_path, remote_name=DVC_PULL_REMOTE)
        os.environ.setdefault('DVC_SITE_CACHE_DIR', str(ROOT / '.dvc' / 'site-cache'))
        if DVC_PULL_ON_KAGGLE:
            run_command('dvc', 'pull', '--force', *DVC_PULL_TARGETS, cwd=ROOT)
else:
    print('SETUP_DVC_SERVICE_ACCOUNT is False; skipping DVC auth bootstrap.')

def run_module(module: str, *args: str) -> None:
    run_command(sys.executable, '-m', module, *args, cwd=ROOT)

print('ROOT                :', ROOT)


$ git clone --branch main https://<TOKEN>@github.com/tlong-ds/news-sentiment-analysis.git /kaggle/working/news-sentiment-analysis


Cloning into '/kaggle/working/news-sentiment-analysis'...


$ /usr/bin/python3 -m pip install -q -r requirements.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.7/279.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 43.2 MB/s eta 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.

$ dvc pull --force data/raw.dvc pipelines/sentiment/dvc.yaml
A       data/raw/
A       data/interim/articles_clean.parquet
A       data/interim/articles_with_sentiment.parquet
A       data/interim/articles_with_sentiment.report.json
A       data/interim/cafef_input.parquet
A       data/interim/daily_aggregation_validation.json
A       data/interim/daily_news_prices.parquet
A       data/interim/input_preparation_report.json
A       data/interim/modeling_ready.parquet
A       data/interim/modeling_ready.report.json
A       data/interim/sentiment_inference_validation.json
A       data/sentiment/article_sentiment_scores.parquet
A       data/sentiment/article_sentiment_scores.report.json
17 files fetched and 16 files added
ROOT                : /kaggle/working/news-sentiment-analysis


## 3 - Volatility Pipeline Parameters

Here we configure the dataset paths and model parameters. These align with the central configurations in `params.yaml`.

In [4]:
PRICES_FILE = ROOT / 'data' / 'raw' / 'prices_VN.csv'
DAILY_NEWS_FILE = ROOT / 'data' / 'interim' / 'daily_news_prices.parquet'
SENTIMENT_FILE = ROOT / 'data' / 'sentiment' / 'article_sentiment_scores.parquet'
ARTICLES_CLEAN_FILE = ROOT / 'data' / 'interim' / 'articles_clean.parquet'
MODELING_READY_FILE = ROOT / 'data' / 'interim' / 'modeling_ready.parquet'

TARGET_TYPE = 'parkinson'
SENTIMENT_THRESHOLD = 0.05
SEQUENCE_LENGTH = 15
TRAIN_END = '2021-12-31'
VAL_END = '2021-12-31'
EPOCHS = 30
BATCH_SIZE = 32

HYBRID_OUTPUT = ROOT / 'data' / 'interim' / 'hybrid_experiment_summary.json'
ROBUSTNESS_OUTPUT = ROOT / 'data' / 'interim' / 'robustness_experiment_summary.json'

for path in [MODELING_READY_FILE.parent, HYBRID_OUTPUT.parent]:
    path.mkdir(parents=True, exist_ok=True)


## 4 - Stage 1: Build the Model Frame

Construct the model frame aligning return volatility (Parkinson) with daily aggregated article sentiments filtered on thresholds and category filters.

In [5]:
!ls data -R

data:
interim  processed  raw  raw.dvc  sentiment

data/interim:
articles_clean.parquet		     daily_news_prices.parquet
articles_with_sentiment.parquet      input_preparation_report.json
articles_with_sentiment.report.json  modeling_ready.parquet
cafef_input.parquet		     modeling_ready.report.json
daily_aggregation_validation.json    sentiment_inference_validation.json

data/processed:

data/raw:
full_data.csv  news_VN_cafef.csv  prices_VN.csv

data/sentiment:
article_sentiment_scores.parquet  article_sentiment_scores.report.json


In [6]:
import pandas as pd
run_module(
    'src.modeling.prepare_model_frame',
    '--prices', str(PRICES_FILE),
    '--daily-news', str(DAILY_NEWS_FILE),
    '--sentiment', str(SENTIMENT_FILE),
    '--articles-clean', str(ARTICLES_CLEAN_FILE),
    '--output-file', str(MODELING_READY_FILE),
    '--target-type', TARGET_TYPE,
    '--sentiment-threshold', str(SENTIMENT_THRESHOLD)
)

df = pd.read_parquet(MODELING_READY_FILE)
print("Modeling Frame Shape:", df.shape)
df.head(3)


$ /usr/bin/python3 -m src.modeling.prepare_model_frame --prices /kaggle/working/news-sentiment-analysis/data/raw/prices_VN.csv --daily-news /kaggle/working/news-sentiment-analysis/data/interim/daily_news_prices.parquet --sentiment /kaggle/working/news-sentiment-analysis/data/sentiment/article_sentiment_scores.parquet --articles-clean /kaggle/working/news-sentiment-analysis/data/interim/articles_clean.parquet --output-file /kaggle/working/news-sentiment-analysis/data/interim/modeling_ready.parquet --target-type parkinson --sentiment-threshold 0.05


2026-05-27 16:17:05,865 - INFO - Saved modeling-ready frame -> /kaggle/working/news-sentiment-analysis/data/interim/modeling_ready.parquet


Modeling Frame Shape: (2498, 31)


,date,close,open,high,low,volume,log_return,abs_return,squared_return,parkinson_vol,...,neutral_share,positive_share,net_sentiment,sentiment_surprise,macro_sentiment,market_sentiment,macro_sentiment_missing,market_sentiment_missing,has_sentiment,has_news
0,2015-01-05,544.45,545.43,549.22,543.78,91834620,NaN,NaN,NaN,0.005978,...,0.006289,0.622642,0.251572,0.270391,0.894394,0.232760,0,0,1,1
1,2015-01-06,549.66,539.08,550.11,538.82,94081890,0.009524,0.009524,0.000091,0.012454,...,0.035714,0.553571,0.142857,-0.119677,0.697682,0.348032,0,0,1,1
2,2015-01-07,552.05,548.44,555.83,548.44,109445780,0.004339,0.004339,0.000019,0.008038,...,0.014286,0.528571,0.071429,-0.092680,0.477909,0.620778,0,0,1,1


## 5 - Stage 2: Fit Econometrics Baseline & Train LSTM

Fits the AR(1)-GARCH(1,1) model on the training period, builds rolling sequences of features (using GARCH conditional variance instead of volatility), and trains the residual-correction LSTM.

In [7]:
run_module(
    'src.modeling.run_experiment',
    '--prices', str(PRICES_FILE),
    '--model-frame', str(MODELING_READY_FILE),
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--train-end', TRAIN_END,
    '--val-end', VAL_END,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--output', str(HYBRID_OUTPUT)
)

summary = load_json(HYBRID_OUTPUT)
print("\n--- Primary Experiment Complete ---")
print("Baseline RMSE:", summary["baseline_metrics"]["rmse"])
print("Hybrid RMSE  :", summary["hybrid_metrics"]["rmse"])
print("Diebold-Mariano p-value:", summary["diebold_mariano"]["p_value"])


$ /usr/bin/python3 -m src.modeling.run_experiment --prices /kaggle/working/news-sentiment-analysis/data/raw/prices_VN.csv --model-frame /kaggle/working/news-sentiment-analysis/data/interim/modeling_ready.parquet --sequence-length 15 --train-end 2021-12-31 --val-end 2021-12-31 --epochs 30 --batch-size 32 --output /kaggle/working/news-sentiment-analysis/data/interim/hybrid_experiment_summary.json


2026-05-27 16:17:10,287 - INFO - Loading modeling-ready frame from /kaggle/working/news-sentiment-analysis/data/interim/modeling_ready.parquet
2026-05-27 16:17:10,314 - INFO - Model frame loaded. Fitting baseline GARCH features...
2026-05-27 16:17:10,776 - INFO - GARCH features added. Building LSTM sequences...
2026-05-27 16:17:12,193 - INFO - LSTM sequences built. Fitting GARCH baseline directly on training window...
2026-05-27 16:17:13,578 - INFO - GARCH diagnostics computed.
2026-05-27 16:17:13,578 - INFO - Training LSTM residual correction model...
2026-05-27 16:17:13,822 - INFO - Training LSTM residual model on device: cuda
2026-05-27 16:17:20,075 - INFO - Epoch 1/30 - loss: 0.001675 - mae: 0.026637
2026-05-27 16:17:20,198 - INFO - Epoch 2/30 - loss: 0.000024 - mae: 0.003514
2026-05-27 16:17:20,325 - INFO - Epoch 3/30 - loss: 0.000020 - mae: 0.002935
2026-05-27 16:17:20,451 - INFO - Epoch 4/30 - loss: 0.000020 - mae: 0.002919
2026-05-27 16:17:20,576 - INFO - Epoch 5/30 - loss: 0.0


=== GARCH Baseline Diagnostics ===
{
  "alpha_plus_beta": 0.9802804977558633,
  "stationary": true,
  "ljung_box_pvalue_lag5": 0.8070209795289172,
  "ljung_box_pvalue_lag10": 0.32635594908512927,
  "no_remaining_arch_effects": true
}

=== Baseline Volatility Forecast Metrics ===
{
  "rmse": 0.005535980227191126,
  "mae": 0.0044494697847878755,
  "mape": 0.7251493351455242
}

=== Hybrid Volatility Forecast Metrics ===
{
  "rmse": 0.004808395025389208,
  "mae": 0.003337523240919168,
  "mape": 0.406072695563656
}

=== Diebold-Mariano Comparative Test ===
{
  "statistic": 5.595912303331793,
  "p_value": 2.1946462203104034e-08,
  "significant_95": true
}

=== Subperiod and Asymmetry Analysis ===
{
  "year_2022": {
    "size": 242,
    "baseline_rmse": 0.006721967831254005,
    "hybrid_rmse": 0.006125004030764103,
    "baseline_mae": 0.005469966679811478,
    "hybrid_mae": 0.004493336193263531
  },
  "year_2023": {
    "size": 247,
    "baseline_rmse": 0.004980374593287706,
    "hybrid_rmse

## 6 - Stage 3: Volatility Robustness Checks

Executes extensive robustness checks including alternative model baselines, subsets of macro vs market news categories, and unweighted sentiment metrics.

In [8]:
run_module(
    'src.modeling.run_robustness',
    '--prices', str(PRICES_FILE),
    '--model-frame', str(MODELING_READY_FILE),
    '--daily-news', str(DAILY_NEWS_FILE),
    '--sentiment', str(SENTIMENT_FILE),
    '--articles-clean', str(ARTICLES_CLEAN_FILE),
    '--sequence-length', str(SEQUENCE_LENGTH),
    '--train-end', TRAIN_END,
    '--val-end', VAL_END,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--output', str(ROBUSTNESS_OUTPUT)
)

robustness = load_json(ROBUSTNESS_OUTPUT)
print("\n--- Robustness Checks Complete ---")
print("Robustness configurations estimated:", list(robustness.keys()))


$ /usr/bin/python3 -m src.modeling.run_robustness --prices /kaggle/working/news-sentiment-analysis/data/raw/prices_VN.csv --model-frame /kaggle/working/news-sentiment-analysis/data/interim/modeling_ready.parquet --daily-news /kaggle/working/news-sentiment-analysis/data/interim/daily_news_prices.parquet --sentiment /kaggle/working/news-sentiment-analysis/data/sentiment/article_sentiment_scores.parquet --articles-clean /kaggle/working/news-sentiment-analysis/data/interim/articles_clean.parquet --sequence-length 15 --train-end 2021-12-31 --val-end 2021-12-31 --epochs 30 --batch-size 32 --output /kaggle/working/news-sentiment-analysis/data/interim/robustness_experiment_summary.json


2026-05-27 16:17:27,132 INFO Running Baseline Model...
2026-05-27 16:17:27,132 INFO Loading modeling-ready frame from /kaggle/working/news-sentiment-analysis/data/interim/modeling_ready.parquet
2026-05-27 16:17:29,278 INFO Training LSTM residual model on device: cuda
2026-05-27 16:17:31,050 INFO Epoch 1/30 - loss: 0.000379 - mae: 0.012822
2026-05-27 16:17:31,170 INFO Epoch 2/30 - loss: 0.000020 - mae: 0.003001
2026-05-27 16:17:31,293 INFO Epoch 3/30 - loss: 0.000019 - mae: 0.002908
2026-05-27 16:17:31,418 INFO Epoch 4/30 - loss: 0.000019 - mae: 0.002917
2026-05-27 16:17:31,543 INFO Epoch 5/30 - loss: 0.000019 - mae: 0.002899
2026-05-27 16:17:31,665 INFO Epoch 6/30 - loss: 0.000020 - mae: 0.002978
2026-05-27 16:17:31,788 INFO Epoch 7/30 - loss: 0.000019 - mae: 0.002893
2026-05-27 16:17:31,913 INFO Epoch 8/30 - loss: 0.000019 - mae: 0.002882
2026-05-27 16:17:32,036 INFO Epoch 9/30 - loss: 0.000019 - mae: 0.002876
2026-05-27 16:17:32,159 INFO Epoch 10/30 - loss: 0.000019 - mae: 0.002876
2


PHASE 7 ROBUSTNESS CHECKS COMPARISON TABLE
| Specification | GARCH RMSE | Hybrid RMSE | GARCH MAE | Hybrid MAE | DM Stat | DM p-value |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **baseline** | 0.005536 | 0.004824 | 0.004449 | 0.003520 | 9.1960 | 0.00000 |
| **threshold_0.10** | 0.005536 | 0.004767 | 0.004449 | 0.003378 | 7.5910 | 0.00000 |
| **threshold_0.15** | 0.005536 | 0.006149 | 0.004449 | 0.004620 | -2.7521 | 0.00592 |
| **garman_klass** | 0.005536 | 0.004773 | 0.004449 | 0.003338 | 6.5545 | 0.00000 |
| **no_sentiment_ablation** | 0.005540 | 0.004899 | 0.004452 | 0.003396 | 4.3517 | 0.00001 |
| **garch_x** | 0.005747 | 0.005734 | 0.004570 | 0.004537 | 1.6377 | 0.10148 |
| **expanding_garch** | 0.005595 | 0.004866 | 0.004504 | 0.003572 | 9.7294 | 0.00000 |

--- Robustness Checks Complete ---
Robustness configurations estimated: ['baseline', 'threshold_0.10', 'threshold_0.15', 'garman_klass', 'no_sentiment_ablation', 'garch_x', 'expanding_garch']
